In [31]:
#import
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader
from tarnet import Tarnet
import sys
from pathlib import Path
project_root = Path("/home/ducvu0904/Documents/Lab/RERUM")
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
sys.path.append("..")
from utils import seed_everything
from metrics import auuc, auqc, lift, krcc

In [32]:
%time train_df = pd.read_csv(r"../dataset/Hillstrom/Men/train_men.csv")
%time test_df =  pd.read_csv(r"../dataset/Hillstrom/Men/test_men.csv")
%time val_df = pd.read_csv(r"../dataset/Hillstrom/Men/val_men.csv")

CPU times: user 27.4 ms, sys: 5 ms, total: 32.5 ms
Wall time: 31.9 ms
CPU times: user 11.4 ms, sys: 5 ms, total: 16.4 ms
Wall time: 16.3 ms
CPU times: user 4.09 ms, sys: 1 ms, total: 5.09 ms
Wall time: 5.06 ms


In [33]:
in_features = ['recency', 'history_segment', 'history', 'mens', 'womens',
       'zip_code', 'newbie', 'channel_Multichannel', 'channel_Phone', 'channel_Web']
label_feature = ['spend']
treatment_feature = ['treatment']

In [34]:
X_train = train_df[in_features].values.astype(float) # type: ignore
y_train = train_df[label_feature].values.astype(float) # type: ignore
t_train = train_df[treatment_feature].values.astype(float) # type: ignore

X_test = test_df[in_features].values.astype(float) # type: ignore
y_test = test_df[label_feature].values.astype(float) # type: ignore
t_test = test_df[treatment_feature].values.astype(float) # type: ignore

X_val = val_df[in_features].values.astype(float) # type: ignore
y_val = val_df[label_feature].values.astype(float) # type: ignore
t_val = val_df[treatment_feature].values.astype(float) # type: ignore

In [35]:
print('X_train[:10]', X_train[:1].astype(float))

X_train[:10] [[-0.21435131  1.6331766   1.0667411   0.90252386 -1.1010233   1.07039981
   1.00043033  2.70003843 -0.88552759 -0.88616046]]


In [36]:
print('y_train[:10]', y_train[:1].astype(float))

y_train[:10] [[0.]]


In [37]:
# Transform to tensor
def to_tensor(arr):
    return torch.tensor(arr, dtype=torch.float32)

x_men_train_t = to_tensor(X_train)
x_men_val_t = to_tensor(X_val)
x_men_test_t = to_tensor(X_test)

y_men_train_t = to_tensor(y_train).reshape(-1, 1)
y_men_val_t = to_tensor(y_val).reshape(-1, 1)
y_men_test_t = to_tensor(y_test).reshape(-1, 1)

# t_train/t_val/t_test cũng tương tự
t_men_train_t = to_tensor(t_train.astype(float)).reshape(-1, 1)
t_men_val_t = to_tensor(t_val.astype(float)).reshape(-1, 1)
t_men_test_t = to_tensor(t_test.astype(float)).reshape(-1, 1)

# Data loader
train_dataset = TensorDataset(x_men_train_t, t_men_train_t, y_men_train_t)
val_dataset = TensorDataset(x_men_val_t, t_men_val_t, y_men_val_t)
test_dataset = TensorDataset(x_men_test_t, t_men_test_t, y_men_test_t)

batch_size = 6400
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory = True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory = True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory = True)

print("-------------------------------------------------------------")
print("✅ Completed transform to tensor ✅")
print(f"Shape of train: x={x_men_train_t.shape}; y={y_men_train_t.shape}; t={t_men_train_t.shape}")
print(f"Shape of val: x={x_men_val_t.shape}; y={y_men_val_t.shape}; t={t_men_val_t.shape}")
print(f"Shape of test: x={x_men_test_t.shape}; y={y_men_test_t.shape}; t={t_men_test_t.shape}")

-------------------------------------------------------------
✅ Completed transform to tensor ✅
Shape of train: x=torch.Size([25567, 10]); y=torch.Size([25567, 1]); t=torch.Size([25567, 1])
Shape of val: x=torch.Size([4262, 10]); y=torch.Size([4262, 1]); t=torch.Size([4262, 1])
Shape of test: x=torch.Size([12784, 10]); y=torch.Size([12784, 1]); t=torch.Size([12784, 1])


In [38]:
epochs = 150
early_stop_metric = "ema_qini"
ema = True
ema_alpha = 0.25
patience = 20
shared_dropout = 0.0
outcome_dropout = 0
shared_hidden = 200
outcome_hidden = 100
early_stop_start = 30
uplift_ranking = 0
response_ranking = 0
print (f" epochs = {epochs}")
print (f" early stop = {early_stop_metric}")
print (f" use ema = {ema}")
print (f" ema alpha = {ema_alpha}")
print (f" patience = {patience}")
print (f" early stop start = {early_stop_start}")

 epochs = 150
 early stop = ema_qini
 use ema = True
 ema alpha = 0.25
 patience = 20
 early stop start = 30


In [ ]:
import io
import optuna
from contextlib import redirect_stdout, redirect_stderr

# Minimize Optuna console noise
optuna.logging.set_verbosity(optuna.logging.WARNING)

# 1. Optuna search config (validation only)
seeds = [412312, 42, 1874, 902745, 1]
n_trials = 50
tpe_sampler_seed = 412312
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def objective(trial):
    grid_lr = trial.suggest_float("lr", 5e-5, 5e-3, log=True)
    grid_wd = trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True)
    grid_outcome_dropout = trial.suggest_float("outcome_dropout", 0.1, 0.5)
    grid_shared_dropout = trial.suggest_float("shared_dropout", 0.0, 0.4)
    grid_shared_hidden = trial.suggest_categorical("shared_hidden", [128, 256, 384, 512])
    grid_outcome_hidden = trial.suggest_categorical("outcome_hidden", [64, 128, 256])

    val_auqc_list = []
    val_ate_err_list = []
    for SEED in seeds:
        seed_everything(SEED)

        tarnet = Tarnet(
            input_dim=x_men_train_t.shape[1],
            epochs=epochs,
            learning_rate=grid_lr,
            weight_decay=grid_wd,
            use_ema=ema,
            ema_alpha=ema_alpha,
            patience=patience,
            shared_hidden=grid_shared_hidden,
            outcome_hidden=grid_outcome_hidden,
            outcome_dropout=grid_outcome_dropout,
            shared_dropout=grid_shared_dropout,
            early_stop_metric=early_stop_metric,
            early_stop_start_epoch=early_stop_start,
        )

        with redirect_stdout(io.StringIO()), redirect_stderr(io.StringIO()):
            tarnet.fit(train_loader, val_loader)

        x_men_val_t_on_device = x_men_val_t.to(device)
        y0_pred, y1_pred = tarnet.predict(x_men_val_t_on_device)

        uplift_pred = (y1_pred - y0_pred).detach().cpu().numpy().flatten()
        y_true = y_men_val_t.detach().cpu().numpy().flatten()
        t_true = t_men_val_t.detach().cpu().numpy().flatten()
        
        current_val_auqc = auqc(y_true, t_true, uplift_pred, bins=100, plot=False)
        ate_pred = uplift_pred.mean()
        ate_true = y_true[t_true == 1].mean() - y_true[t_true == 0].mean()
        current_val_ate_err = abs(ate_pred - ate_true)

        val_auqc_list.append(current_val_auqc)
        val_ate_err_list.append(current_val_ate_err)

    # Calculate aggregated metrics across the 5 validation seeds
    mean_auqc = float(np.mean(val_auqc_list))
    std_auqc = float(np.std(val_auqc_list))
    mean_ate_err = float(np.mean(val_ate_err_list))

    # Apply penalty for instability and miscalibration
    penalty_std = std_auqc * 2.0

    final_score = mean_auqc - penalty_std

    trial.set_user_attr("mean_val_auqc", mean_auqc)
    trial.set_user_attr("std_Val_auqc", std_auqc)
    trial.set_user_attr("mean_val_ate_err", mean_ate_err)
    trial.set_user_attr("final_score", final_score) 
    return final_score

def print_trial_callback(study, trial):
    value = float(trial.value) if trial.value is not None else float("nan")
    best_trial = study.best_trial
    best_value = float(best_trial.value) if best_trial.value is not None else float("nan")
    print(
        f"Finished trial {trial.number}: val loss: {value:.4f} - "
        f"with hyperparameters: {trial.params} | "
        f"best trial: {best_trial.number} loss: {best_value:.4f}",
        flush=True
    )

sampler = optuna.samplers.TPESampler(seed=tpe_sampler_seed)
study = optuna.create_study(direction="maximize", sampler=sampler)
study.optimize(objective, n_trials=n_trials, show_progress_bar=True, callbacks=[print_trial_callback])

trial_rows = []
for t in study.trials:
    if t.value is None:
        continue
    trial_rows.append({
        "trial": t.number,
        "lr": round(float(t.params["lr"]), 4),
        "weight_decay": round(float(t.params["weight_decay"]), 4),
        "shared_hidden": int(t.params["shared_hidden"]),
        "outcome_hidden": int(t.params["outcome_hidden"]),
        "shared_dropout": round(float(t.params["shared_dropout"]), 3),
        "outcome_dropout": round(float(t.params["outcome_dropout"]), 3),
        "mean_val_auqc": float(t.value),
        "std_Val_auqc": float(t.user_attrs.get("std_Val_auqc", np.nan))
    })

df_grid = pd.DataFrame(trial_rows).sort_values("mean_val_auqc", ascending=True).reset_index(drop=True)
best_params = study.best_params
best_cfg = pd.Series({
    "lr": float(best_params["lr"]),
    "weight_decay": float(best_params["weight_decay"]),
    "shared_hidden": int(best_params["shared_hidden"]),
    "outcome_hidden": int(best_params["outcome_hidden"]),
    "shared_dropout": float(best_params["shared_dropout"]),
    "outcome_dropout": float(best_params["outcome_dropout"]),
    "mean_Val_auqc": float(study.best_value),
    "std_Val_auqc": float(study.best_trial.user_attrs.get("std_Val_auqc", np.nan))
})

  0%|          | 0/50 [00:00<?, ?it/s]

Finished trial 0: val loss: 0.7420 - with hyperparameters: {'lr': 0.00045041481661314923, 'weight_decay': 3.2543813547868006e-05, 'outcome_dropout': 0.26330970370677254, 'shared_dropout': 0.17735402643744136, 'shared_hidden': 512, 'outcome_hidden': 256} | best trial: 0 loss: 0.7420


Best trial: 0. Best value: 0.742028:   2%|▏         | 1/50 [01:12<59:21, 72.69s/it]

Finished trial 1: val loss: 0.7852 - with hyperparameters: {'lr': 0.00020955060428245223, 'weight_decay': 4.661152760287686e-05, 'outcome_dropout': 0.16868166430648263, 'shared_dropout': 0.086292445337752, 'shared_hidden': 256, 'outcome_hidden': 256} | best trial: 1 loss: 0.7852


Best trial: 1. Best value: 0.785203:   4%|▍         | 2/50 [02:24<57:39, 72.08s/it]

...

Best trial: 34. Best value: 0.847736:  96%|█████████▌| 48/50 [1:01:40<03:14, 97.46s/it] 

Finished trial 48: val loss: 0.8303 - with hyperparameters: {'lr': 0.0039621853619127975, 'weight_decay': 0.006795005588244638, 'outcome_dropout': 0.4366735206563256, 'shared_dropout': 0.06684578006272696, 'shared_hidden': 256, 'outcome_hidden': 128} | best trial: 34 loss: 0.8477


Best trial: 34. Best value: 0.847736:  98%|█████████▊| 49/50 [1:02:52<01:29, 89.73s/it]

Finished trial 49: val loss: 0.7526 - with hyperparameters: {'lr': 0.0021111473966806474, 'weight_decay': 0.0001505514189704889, 'outcome_dropout': 0.3707931034233204, 'shared_dropout': 0.09916001631081445, 'shared_hidden': 128, 'outcome_hidden': 64} | best trial: 34 loss: 0.8477


Best trial: 34. Best value: 0.847736: 100%|██████████| 50/50 [1:04:03<00:00, 76.87s/it]


In [40]:
from IPython.display import display

if 'study' not in globals():
    print('Run Cell 10 (Optuna tuning) first.')
else:
    print(f"Best trial: {study.best_trial.number}")
    print(f"Best mean_Val_AUQC: {study.best_value:.6f}")
    print(f"Best params: {study.best_params}")

if 'best_cfg' in globals():
    print('\nBest config table:')
    display(best_cfg.to_frame().T)
else:
    print('\nbest_cfg not found.')

if 'df_grid' in globals():
    print('\nTop 10 trials:')
    display(df_grid.head(10))
else:
    print('\ndf_grid not found.')

if 'df_results' in globals():
    print('\nPer-seed test results:')
    display(df_results)
    print('\nTest metrics mean ± std:')
    display(df_results.drop(columns='Seed').agg(['mean', 'std']))

Best trial: 34
Best mean_Val_AUQC: 0.847736
Best params: {'lr': 0.0023466281055962448, 'weight_decay': 0.008823140898160494, 'outcome_dropout': 0.3894592778283563, 'shared_dropout': 0.0871028952795896, 'shared_hidden': 256, 'outcome_hidden': 64}

Best config table:


,lr,weight_decay,shared_hidden,outcome_hidden,shared_dropout,outcome_dropout,mean_Val_auqc,std_Val_auqc
0,0.002347,0.008823,256.0,64.0,0.087103,0.389459,0.847736,0.008134



Top 10 trials:


,trial,lr,weight_decay,shared_hidden,outcome_hidden,shared_dropout,outcome_dropout,mean_val_auqc,std_Val_auqc
0,10,0.0001,0.0100,128,128,0.399,0.476,0.373470,0.147158
1,46,0.0001,0.0023,256,256,0.091,0.339,0.642288,0.051849
2,40,0.0013,0.0000,128,128,0.163,0.124,0.715431,0.045464
3,19,0.0001,0.0030,256,128,0.023,0.310,0.724902,0.032864
4,29,0.0016,0.0002,512,64,0.189,0.453,0.731309,0.026666
5,4,0.0004,0.0000,128,256,0.170,0.271,0.739707,0.035660
6,0,0.0005,0.0000,512,256,0.177,0.263,0.742028,0.023437
7,45,0.0004,0.0010,512,64,0.128,0.270,0.748977,0.022385
8,14,0.0022,0.0003,512,64,0.137,0.424,0.749836,0.024850
9,8,0.0034,0.0001,512,256,0.281,0.321,0.751908,0.022459



Per-seed test results:


,Seed,AUUC,AUQC,Lift,KRCC,ATE_Err
0,412312,0.606361,0.604859,1.195012,0.000862,0.151801
1,42,0.575012,0.572734,0.595819,0.038140,0.190205
2,1874,0.534799,0.532669,0.136236,0.035423,0.189581
3,902745,0.747138,0.746663,1.154948,0.164809,0.105935
4,1,0.617501,0.614011,0.931758,0.053076,0.035142



Test metrics mean ± std:


,AUUC,AUQC,Lift,KRCC,ATE_Err
mean,0.616162,0.614187,0.802755,0.058462,0.134533
std,0.079947,0.080623,0.442046,0.062450,0.065430


In [41]:
import pandas as pd
import numpy as np
import torch

# 1. Evaluate selected config on test set (after tuning)
seeds = [412312, 42, 1874, 902745, 1]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
all_runs = []

if 'best_cfg' not in globals():
    raise ValueError("best_cfg not found. Run grid-search cell first.")

best_lr = float(best_cfg['lr'])
best_wd = float(best_cfg['weight_decay'])
best_shared_hidden = int(best_cfg['shared_hidden'])
best_outcome_hidden = int(best_cfg['outcome_hidden'])
best_shared_dropout = float(best_cfg['shared_dropout'])
best_outcome_dropout = float(best_cfg['outcome_dropout'])

print("Evaluating on test with best validation config:")
print(f"  lr={best_lr:.1e}, weight_decay={best_wd:.1e}")
print(f"  shared_hidden={best_shared_hidden}, outcome_hidden={best_outcome_hidden}")
print(f"  shared_dropout={best_shared_dropout:.3f}, outcome_dropout={best_outcome_dropout:.3f}")
print(f"Number of seeds: {len(seeds)}")

# 2. Loop over seeds for robust test evaluation
for SEED in seeds:
    seed_everything(SEED)

    tarnet = Tarnet(
        input_dim=x_men_train_t.shape[1],
        epochs=epochs,
        learning_rate=best_lr,
        weight_decay=best_wd,
        use_ema=ema,
        ema_alpha=ema_alpha,
        patience=patience,
        shared_hidden=best_shared_hidden,
        outcome_hidden=best_outcome_hidden,
        outcome_dropout=best_outcome_dropout,
        shared_dropout=best_shared_dropout,
        early_stop_metric=early_stop_metric,
        early_stop_start_epoch=early_stop_start,
    )

    tarnet.fit(train_loader, val_loader)

    # Test prediction
    x_men_test_t_on_device = x_men_test_t.to(device)
    y0_pred, y1_pred = tarnet.predict(x_men_test_t_on_device)

    uplift_pred = (y1_pred - y0_pred).detach().cpu().numpy().flatten()
    y_true = y_men_test_t.detach().cpu().numpy().flatten()
    t_true = t_men_test_t.detach().cpu().numpy().flatten()

    # ATE error
    ate_pred = uplift_pred.mean()
    ate_true = y_true[t_true == 1].mean() - y_true[t_true == 0].mean()

    all_runs.append({
        'Seed': SEED,
        'AUUC': auuc(y_true, t_true, uplift_pred, bins=100, plot=False),
        'AUQC': auqc(y_true, t_true, uplift_pred, bins=100, plot=False),
        'Lift': lift(y_true, t_true, uplift_pred, h=0.3),
        'KRCC': krcc(y_true, t_true, uplift_pred, bins=100),
        'ATE_Err': abs(ate_pred - ate_true)
    })
    print(f"  Done Seed {SEED}")

# 3. Aggregate final test metrics
df_results = pd.DataFrame(all_runs)

print("\n" + "=" * 85)
print(f"{'PER-SEED DETAILS (TEST SET)':^85}")
print("=" * 85)
print(df_results.to_string(index=False, formatters={
    'AUUC': '{:,.4f}'.format,
    'AUQC': '{:,.4f}'.format,
    'Lift': '{:,.4f}'.format,
    'KRCC': '{:,.4f}'.format,
    'ATE_Err': '{:,.4f}'.format
}))

mean_res = df_results.drop(columns='Seed').mean()
std_res = df_results.drop(columns='Seed').std()

print("=" * 85)
print(f"{'TEST SUMMARY (MEAN ± STD)':^85}")
print("-" * 85)
for metric in ['AUUC', 'AUQC', 'Lift', 'KRCC', 'ATE_Err']:
    print(f"{metric:<10}: {mean_res[metric]:.4f} ± {std_res[metric]:.4f}")
print("=" * 85)

Evaluating on test with best validation config:
  lr=2.3e-03, weight_decay=8.8e-03
  shared_hidden=256, outcome_hidden=64
  shared_dropout=0.087, outcome_dropout=0.389
Number of seeds: 5
🔃🔃🔃Begin training Tarnet🔃🔃🔃
📊 Early Stop Metric: EMA_QINI
📊 Early Stop Start Epoch: 31
📊 Strategy: Best EMA Qini (alpha=0.25)
   Restore to epoch with highest smoothed (EMA) Qini score
   Patience: 20 epochs
Epoch 1/150 | Base Loss: 267.1193 | Total Loss: 267.1193 | Val Loss: 497.5274 | Val Qini: 0.7114 | EMA Qini: 0.7114 | Best EMA: 0.7114 ⭐ NEW BEST EMA
Epoch 2/150 | Base Loss: 370.7234 | Total Loss: 370.7234 | Val Loss: 495.6738 | Val Qini: 0.7842 | EMA Qini: 0.7296 | Best EMA: 0.7296 ⭐ NEW BEST EMA
Epoch 3/150 | Base Loss: 355.9018 | Total Loss: 355.9018 | Val Loss: 495.6649 | Val Qini: 0.8588 | EMA Qini: 0.7619 | Best EMA: 0.7619 ⭐ NEW BEST EMA
Epoch 4/150 | Base Loss: 435.6848 | Total Loss: 435.6848 | Val Loss: 495.9754 | Val Qini: 0.8415 | EMA Qini: 0.7818 | Best EMA: 0.7818 ⭐ NEW BEST EMA
Epoch